# NVIDIA NeMo

A self-contained refresher on **NVIDIA NeMo** — NVIDIA's open-source toolkit for building, training, and serving **conversational-AI** models (ASR, TTS, and speech/LLM NLP).

**Domain:** Speech & Audio  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**NeMo** is a PyTorch-Lightning-based framework that packages NVIDIA's speech and language research into reusable **neural modules** plus a catalog of **pretrained checkpoints** you can pull straight from NGC / Hugging Face. Its center of gravity is **speech**: state-of-the-art **ASR** (Conformer, Citrinet, QuartzNet, and the newer **Parakeet** / **Canary** families), **TTS** (FastPitch + HiFi-GAN, RAD-TTS), speaker tasks (diarization, verification with TitaNet), plus an NLP/LLM stack (Megatron) for the same teams that own the GPUs.

**The problem it solves.** Production speech models are a pile of fiddly pieces: audio front-ends, tokenizers, CTC/RNN-T/attention decoders, schedulers, mixed precision, multi-GPU sharding, and checkpoint plumbing. NeMo gives you all of that pre-wired behind a `Model` class so a one-liner `from_pretrained(...)` gives you a transcriber, and a YAML + `Trainer.fit()` fine-tunes it on your own data — on one GPU or hundreds, with the *same* config.

**Reach for it when** you want best-in-class transcription/synthesis with minimal glue, when you'll fine-tune on domain audio, or when you're already in the NVIDIA stack (Triton/Riva deployment, multi-GPU training). **Skip it when** you just need a quick transcript on CPU (use `faster-whisper`), when you can't stomach a large CUDA/PyTorch dependency, or when your problem isn't speech/LLM-shaped — NeMo is heavyweight and GPU-first.

## 2. Mental Model

Think of NeMo as **"Keras for conversational AI, opinionated toward NVIDIA hardware."** Every task is a `nemo.collections.<asr|tts|nlp>` **`Model`** that bundles three things into one object:

```
          ┌───────────────────────────  NeMo Model (.nemo file)  ───────────────────────────┐
  audio → │  preprocessor (mel front-end)  →  encoder (Conformer/…)  →  decoder (CTC/RNN-T)  │ → text
          │  ▲ config (Hydra/YAML)            ▲ weights                  ▲ tokenizer/vocab    │
          └─────────────────────────────────────────────────────────────────────────────────┘
                 the whole box trains with PyTorch-Lightning `Trainer.fit(model)`
```

A **`.nemo`** file is just a tarball: the YAML config + the weights + the tokenizer. That single artifact is reproducible and self-describing — `restore_from('foo.nemo')` rebuilds the exact graph. Data flows in through a **manifest** (one JSON object per line: `audio_filepath`, `duration`, `text`). So the whole workflow is: *manifest in → Model (config + weights) → text out*, with Lightning handling the training loop you'd otherwise hand-write.

## 3. Key Concepts

- **Neural Module / `Model`.** A self-contained `LightningModule` subclass (e.g. `EncDecCTCModelBPE`, `FastPitchModel`) bundling preprocessor + encoder + decoder + loss. You mostly use it, rarely rebuild it.
- **`.nemo` checkpoint.** A tarball of `{config.yaml, model_weights, tokenizer}`. `Model.restore_from(path)` or `Model.from_pretrained(name)` loads it; `model.save_to(path)` writes it.
- **Manifest.** Newline-delimited JSON (JSONL). Each line: `{"audio_filepath": ..., "duration": ..., "text": ...}`. This is *the* data interface for ASR/TTS train and eval.
- **Decoders — CTC vs RNN-T vs attention.** **CTC** is fast, non-autoregressive, emits a per-frame distribution over vocab + a **blank** token (greedy decode = collapse repeats, drop blanks). **RNN-T** (Transducer) is streaming-friendly and usually more accurate but autoregressive. Canary uses an attention encoder-decoder for multitask ASR+translation.
- **Preprocessor / mel front-end.** `AudioToMelSpectrogramPreprocessor` turns raw 16 kHz audio into log-mel features — the encoder never sees raw waveforms.
- **Hydra config.** Everything (model arch, optimizer, data) is a YAML tree you override on the CLI (`model.optim.lr=1e-3`). Reproducibility lives in the config.
- **PyTorch Lightning `Trainer`.** Owns the loop: mixed precision (`precision='bf16'`), gradient accumulation, multi-GPU/multi-node (`devices`, `strategy='ddp'`) — you don't write training loops.
- **NGC / Hugging Face catalog.** Pretrained model names like `stt_en_conformer_ctc_small` or `nvidia/parakeet-tdt-1.1b` resolve to downloadable `.nemo` checkpoints.
- **Riva.** NVIDIA's production inference server; NeMo models export to it for low-latency streaming deployment.

## 4. Setup

Real NeMo is heavy (PyTorch + CUDA + a long dependency list) and GPU-first:

```bash
# CPU/GPU install — pulls a large dependency tree; do this in a clean venv
pip install nemo_toolkit['asr']        # or ['all'] for asr+tts+nlp
```

Then a transcript is genuinely a one-liner:

```python
import nemo.collections.asr as nemo_asr
asr = nemo_asr.models.ASRModel.from_pretrained("stt_en_conformer_ctc_small")
print(asr.transcribe(["audio.wav"]))
```

The worked examples below illustrate NeMo's **core mechanics with plain NumPy** so they run anywhere on CPU in milliseconds; the final example shows the real `from_pretrained` call shape, **gated** behind an env var + import check so this notebook always executes top-to-bottom.

In [ ]:
import json
import os

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

## 5. Worked Examples

### Example 1 — CTC greedy decoding (the heart of a NeMo CTC ASR model)

A NeMo CTC model's encoder emits, per audio frame, a probability distribution over the vocabulary **plus a special blank token**. Greedy decoding is exactly two rules: **(1) take the arg-max token per frame, (2) collapse consecutive duplicates, then (3) remove blanks.** That collapse step is why "hello" with a held vowel still decodes to one word. Here it is from scratch.

In [ ]:
# A tiny vocab: index 0 is the CTC blank, the rest are characters.
BLANK = 0
vocab = ["<blank>", "h", "e", "l", "o", " "]


def ctc_greedy_decode(logits, vocab, blank=BLANK):
    """logits: (T, V) array of per-frame scores -> decoded string."""
    ids = logits.argmax(axis=-1)          # (1) best token per frame
    out = []
    prev = None
    for i in ids:
        if i != prev and i != blank:      # (2) collapse repeats, (3) drop blanks
            out.append(vocab[i])
        prev = i
    return "".join(out), ids


# Hand-built frame token stream for "hello": repeats + blanks (as a real encoder emits).
frame_tokens = [1, 1, BLANK, 2, BLANK, 3, 3, BLANK, 3, 4, 4]  # h h _ e _ l l _ l o o
T, V = len(frame_tokens), len(vocab)
logits = np.full((T, V), -5.0)
logits[np.arange(T), frame_tokens] = 5.0   # make the intended token win each frame

text, ids = ctc_greedy_decode(logits, vocab)
print("raw arg-max frames :", [vocab[i] for i in ids])
print("decoded transcript :", repr(text))

### Example 2 — The mel front-end and the NeMo manifest

Two things every NeMo speech job needs. **(a)** The encoder never sees raw audio — a preprocessor (`AudioToMelSpectrogramPreprocessor`) converts 16 kHz waveforms into **log-mel** frames. Below is a minimal STFT → mel-ish power → log pipeline so you can see the `(time, n_mels)` feature shape the encoder consumes. **(b)** Data reaches NeMo through a **manifest**: one JSON object per line. We build and read one back.

In [ ]:
# (a) Minimal log-mel-style front-end on a synthetic 0.5 s, 16 kHz tone.
sr, dur = 16_000, 0.5
t = np.arange(int(sr * dur)) / sr
audio = 0.6 * np.sin(2 * np.pi * 220 * t) + 0.05 * rng.standard_normal(t.size)

win, hop, n_mels = 400, 160, 64           # 25 ms window, 10 ms hop -> ~16 ms frames
frames = np.lib.stride_tricks.sliding_window_view(audio, win)[::hop]
window = np.hanning(win)
spec = np.abs(np.fft.rfft(frames * window, axis=1)) ** 2          # power spectrogram
# Crude triangular-ish mel pooling: average adjacent FFT bins down to n_mels bands.
bands = np.array_split(spec, n_mels, axis=1)
mel = np.stack([b.mean(axis=1) for b in bands], axis=1)
log_mel = np.log(mel + 1e-9)

print("audio samples      :", audio.shape)
print("log-mel features   :", log_mel.shape, "(time_frames, n_mels)")
print("feature value range:", f"{log_mel.min():.2f} .. {log_mel.max():.2f}")

In [ ]:
# (b) Build a NeMo-style manifest (JSONL) and read it back.
examples = [
    {"audio_filepath": "/data/utt1.wav", "duration": dur, "text": "hello"},
    {"audio_filepath": "/data/utt2.wav", "duration": 1.2, "text": "hello world"},
]
manifest = "\n".join(json.dumps(e) for e in examples)
print("manifest.jsonl:\n" + manifest)

parsed = [json.loads(line) for line in manifest.splitlines()]
total = sum(e["duration"] for e in parsed)
print(f"\nparsed {len(parsed)} utterances, {total:.2f}s total audio")

### Example 3 — Real NeMo ASR inference (gated)

This is the actual NeMo workflow. It's gated behind `RUN_NEMO=1` **and** an import check, because `nemo_toolkit` is a large GPU-first dependency we don't assume is installed — so the notebook still runs end-to-end without it. Set the env var in an environment where NeMo is installed to download `stt_en_conformer_ctc_small` (~50 MB) and transcribe a WAV.

In [ ]:
if os.getenv("RUN_NEMO") == "1":
    try:
        import nemo.collections.asr as nemo_asr

        # One-liner load of a small pretrained CTC model from the NGC/HF catalog.
        asr = nemo_asr.models.ASRModel.from_pretrained("stt_en_conformer_ctc_small")
        # transcribe() accepts a list of audio paths; returns hypotheses.
        hyps = asr.transcribe(["sample.wav"])
        print("transcript:", hyps[0].text if hasattr(hyps[0], "text") else hyps[0])
    except Exception as exc:  # pragma: no cover - depends on optional heavy dep
        print("NeMo path not runnable here:", type(exc).__name__, exc)
else:
    print("Skipped real NeMo inference. Install nemo_toolkit['asr'] and set RUN_NEMO=1.")
    print("Call shape:")
    print("  import nemo.collections.asr as nemo_asr")
    print("  asr = nemo_asr.models.ASRModel.from_pretrained('stt_en_conformer_ctc_small')")
    print("  asr.transcribe(['audio.wav'])")

## 6. Gotchas & Pitfalls

- **It's GPU-first and heavy.** `nemo_toolkit['all']` drags in a large CUDA/PyTorch tree. CPU inference works for small models but training is effectively GPU-only. Don't reach for NeMo to grab one transcript on a laptop — use `faster-whisper`.
- **Sample rate must match the model.** Almost all NeMo ASR expects **16 kHz mono**. Feed 44.1 kHz or stereo and you get garbage or a shape error — resample first.
- **`.nemo` ≠ raw `.ckpt`.** A `.nemo` tarball carries config + tokenizer + weights and restores the exact graph; a bare Lightning `.ckpt` does not. Use `restore_from` / `save_to`, not manual `torch.load`, unless you know why.
- **Manifest paths and `duration` matter.** Bucketing and batching rely on `duration`; a missing/wrong value or a non-existent `audio_filepath` fails late and confusingly. Validate the manifest first.
- **CTC vs RNN-T decoding differ.** Greedy CTC is the simple collapse shown above; RNN-T/Transducer decoding is autoregressive and has its own beam/streaming config. Don't assume one model's decode code works for the other.
- **Version churn.** NeMo's APIs and config schema move fast across releases; tutorials pin a version. A config from an old NeMo may not load in a new one — match the toolkit version to the checkpoint.
- **Hydra overrides are exact.** A typo'd override key (`model.optim.lr` vs `model.optimizer.lr`) is silently ignored or errors; check the printed resolved config.
- **Mixed precision quirks.** `bf16` is safest on Ampere+; `fp16` can NaN on some ASR losses. Prefer `bf16` when available.

## 7. When to Use vs Alternatives

| Option | Trade-off vs NeMo |
|---|---|
| **faster-whisper / WhisperX** | Far lighter, great CPU/GPU transcription with one pip install and no manifest. Pick it for quick/offline transcription. NeMo wins for fine-tuning, RNN-T streaming, and the NVIDIA deployment path. |
| **Hugging Face Transformers (Wav2Vec2, Whisper)** | Broader ecosystem, simpler API, easy CPU use. NeMo offers stronger out-of-the-box English ASR (Parakeet/Canary top leaderboards) and first-class multi-GPU training + Riva export. |
| **SpeechBrain** | Friendlier, more hackable research framework; lighter install. NeMo scales further (multi-node Megatron, production Riva) and has NVIDIA-tuned SOTA checkpoints. |
| **Coqui TTS / 🐸** | Simpler dedicated TTS. NeMo's FastPitch+HiFi-GAN is competitive and shares one framework with your ASR/NLP, but is heavier. |
| **Roll-your-own PyTorch** | Maximum control. You re-implement front-ends, decoders, multi-GPU, and checkpointing that NeMo gives for free — only worth it for genuinely novel architectures. |

**Bottom line:** choose NeMo when you're fine-tuning speech models, want SOTA NVIDIA checkpoints, or are deploying through the NVIDIA stack (Triton/Riva, multi-GPU). For a fast transcript on modest hardware, reach for faster-whisper or HF Transformers instead.

## 8. Resources

- **Official docs** — https://docs.nvidia.com/nemo-framework/user-guide/latest/
- **GitHub (NVIDIA/NeMo)** — https://github.com/NVIDIA/NeMo
- **ASR tutorials & notebooks** — https://github.com/NVIDIA/NeMo/tree/main/tutorials/asr
- **Pretrained model catalog (Hugging Face)** — https://huggingface.co/nvidia (e.g. `parakeet-tdt-1.1b`, `canary-1b`)
- **Parakeet ASR overview** — https://developer.nvidia.com/blog/pushing-the-boundaries-of-speech-recognition-with-nvidia-nemo-parakeet-asr-models/
- **Riva deployment** — https://docs.nvidia.com/deeplearning/riva/user-guide/docs/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
REQUIRED = ("audio_filepath", "duration", "text")


def parse_manifest(text):
    ...


def write_nemo(config, weights, tokenizer):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE